# Madison, Wisconsin — Land Value Tax Policy Analysis

Madison is the capital of Wisconsin and the seat of **Dane County** (county-equivalent Census FIPS **55025**). The City of Madison assesses all locally taxable real property itself; the ~1% of the base that is manufacturing property is valued by the Wisconsin Department of Revenue. **Wisconsin assesses at 100% of market value** (assessment ratio = 1.0), and the assessor's roll reports a **land value / improvement value split for every parcel** — exactly what a split-rate model needs.

## Policy modeled (custom-scoped Landslide run)

| Parameter | Value |
|---|---|
| Reform | **Split-rate (two-rate) property tax** |
| Land : improvement ratio | **4 : 1** |
| Scope | **City of Madison MUNICIPAL levy ONLY** |
| Exemptions | Preserve all existing exemptions |
| Revenue target | Revenue-neutral to the City of Madison's certified **municipal** levy |

### Why "municipal portion only" matters in Wisconsin

A Madison property-tax bill is a **stacked** mill rate: City of Madison + Dane County + Madison Metropolitan School District (MMSD) + Madison Area Technical College (MATC / Madison College) + the state forestry mill, less the state School Levy Tax Credit and First Dollar Credit. This model applies the split-rate **only to the City of Madison's own portion** of the rate and holds every other jurisdiction's levy unchanged. The City portion is roughly **one-third** of the consolidated net rate.

**City of Madison municipal rate (2026 budget): 7.0032 mills** ($7.0032 per $1,000 of assessed value), down 4.12% from 7.2823 mills in 2025. On the average home ($481,300, 2025 roll) the City portion of the bill is ~$3,371.
Sources: City of Madison 2026 Executive Operating Budget; District 10 "Let's Talk Budget: 2026 Property Taxes" (2025-12-13).

### Revenue target (the official figure this model is gated against)

**City of Madison certified 2026 municipal property-tax levy = $326,666,400** (2026 Executive Operating Budget; a +2.7% / +$8.6M increase over 2025). The city computes its mill rate as levy ÷ net-taxable value: $326,666,400 ÷ **$46,627,007,300** net-taxable value ≈ **7.00 mills** — matching the published rate to within 0.05% (an independent check that the rate/base/levy triangle is right).

### Assessment-roll timing note (important for the revenue gate)

The certified 7.0032-mill rate and the $326.67M levy were struck on the **January 1, 2025** assessment roll (net-taxable base $46.627B, excludes TIF increment). The City's open-data parcel layer, however, is the **newer January 1, 2026 roll** (average single-family value $510,521 vs. the 2025 roll's $481,300 — ~6% higher). Because the newer roll's gross taxable value ($51.67B) is a different year and includes TIF-increment value, applying 7.0032 mills to it directly would overshoot the certified levy by ~11%. Instead — following the repo's **Baltimore "derive the rate from the certified levy"** pattern — the model spreads the **certified $326,666,400 municipal levy** across the current taxable roll, yielding an **effective 6.32 mills** on the 2026 roll. This makes the model **revenue-neutral to the City's actual municipal levy by construction**, and the revenue-neutral split-rate result (who pays more / less, and by what %) is independent of which roll year is used.

## Data source

- **City of Madison Tax Parcels (Assessor Property Information)** — ArcGIS FeatureServer, hosted open data (updated daily):
  `https://maps.cityofmadison.com/arcgis/rest/services/Public/OPEN_DATA2/FeatureServer/0`
  Key fields: `CurrentLand`, `CurrentImpr`, `CurrentTotal` (100%-of-market assessed values), `PropertyClass` (WI statutory class), `PropertyUse` (granular use), `ExemptionType`, `TIFDistrict`, `Parcel` (parcel id), polygon geometry.

## Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `CurrentLand` | 100% market value (WI) |
| Improvement value | `CurrentImpr` | Buildings/structures, 100% market |
| Total value | `CurrentTotal` | = Land + Improvement |
| Use / class | `PropertyClass` + `PropertyUse` | WI class + granular use string |
| Exemption | `PropertyClass == 'Exempt'` | fully-exempt held out |
| Parcel id | `Parcel` | for map / joins |

Assessment ratio: **1.0** (Wisconsin, 100% of market value). Millage source: City of Madison 2026 Executive Operating Budget (certified municipal levy + rate).


In [ ]:
import os
import sys
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon, MultiPolygon

sys.path.insert(0, str(Path("../..").resolve()))
REPO_ROOT = Path("../..").resolve()
load_dotenv(REPO_ROOT / ".env")

from lvt.lvt_utils import (
    calculate_current_tax,
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# ---- Constants ----
CITY_NAME = "madison"
STATE_FIPS = "55"          # Wisconsin
COUNTY_FIPS = "025"        # Dane County  -> FIPS 55025
MODEL_TYPE = "split_rate:4.0"
LAND_IMPROVEMENT_RATIO = 4.0

# City of Madison certified 2026 MUNICIPAL levy (revenue-neutral target).
# Source: City of Madison 2026 Executive Operating Budget.
CERTIFIED_CITY_LEVY = 326_666_400.0
# Published statutory municipal rate (2026 budget), for reporting/validation only.
PUBLISHED_CITY_MILLAGE = 7.0032
# Official net-taxable base the published rate was struck on (Jan 1 2025 roll).
OFFICIAL_NET_TAXABLE_2025ROLL = 46_627_007_300.0

# Parcel-map identity columns
PARCEL_ID_COL = "Parcel"
OWNER_NAME_COL = None
OWNER_ADDRESS_COL = None
PARCEL_URL_TEMPLATE = None   # Madison assessor lookup is a session-based search; no stable deep link

# ArcGIS source
BASE_URL = "https://maps.cityofmadison.com/arcgis/rest/services/Public/OPEN_DATA2/FeatureServer"
LAYER_ID = 0

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARCEL_PATH = DATA_DIR / "parcels.gpq"

print(f"City: {CITY_NAME}  FIPS: {STATE_FIPS+COUNTY_FIPS}  Model: {MODEL_TYPE}")
print(f"Certified municipal levy target: ${CERTIFIED_CITY_LEVY:,.0f}")
print(f"Published municipal rate: {PUBLISHED_CITY_MILLAGE} mills")
# Independent rate/base/levy check:
_chk = PUBLISHED_CITY_MILLAGE * OFFICIAL_NET_TAXABLE_2025ROLL / 1000
print(f"Rate x official 2025 net-taxable base = ${_chk:,.0f}  (vs certified levy: {(_chk/CERTIFIED_CITY_LEVY-1)*100:+.2f}%)")


In [ ]:
# Section 2 — Fetch / load parcel data (geometry + assessment fields), cache to data/parcels.gpq

FIELDS = [
    "Parcel", "Address", "PropertyClass", "PropertyUse", "PropertyUseCode",
    "CurrentLand", "CurrentImpr", "CurrentTotal", "NetTaxes", "TotalTaxes",
    "ExemptionType", "TIFDistrict", "LotSize", "YearBuilt", "TotalLivingArea",
    "TotalDwellingUnits",
]

def _rings_to_geom(rings):
    if not rings:
        return None
    polys = []
    for ring in rings:
        if len(ring) >= 4:
            polys.append(Polygon(ring))
    if not polys:
        return None
    # Largest ring is the shell; treat remaining as separate polygons (assessor rings
    # are simple parcel outlines — holes are rare). Union via MultiPolygon.
    return polys[0] if len(polys) == 1 else MultiPolygon([p for p in polys if p.is_valid or True])

def fetch_parcels(base_url, layer_id, fields, chunk=2000):
    query = f"{base_url}/{layer_id}/query"
    total = requests.get(query, params={"f": "json", "where": "1=1",
                                        "returnCountOnly": "true"}, timeout=60).json()["count"]
    print(f"Total parcels to fetch: {total:,}")
    rows, geoms, offset = [], [], 0
    while offset < total:
        params = {
            "f": "json", "where": "1=1", "outFields": ",".join(fields),
            "returnGeometry": "true", "outSR": 4326,
            "resultOffset": offset, "resultRecordCount": chunk,
        }
        r = requests.get(query, params=params, timeout=180)
        r.raise_for_status()
        feats = r.json().get("features", [])
        if not feats:
            break
        for f in feats:
            rows.append(f["attributes"])
            geoms.append(_rings_to_geom(f.get("geometry", {}).get("rings")))
        offset += len(feats)
        if offset % 20000 < chunk:
            print(f"  ...{offset:,}")
        if len(feats) < chunk:
            break
        time.sleep(0.1)
    gdf = gpd.GeoDataFrame(pd.DataFrame(rows), geometry=geoms, crs="EPSG:4326")
    return gdf

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache: {PARCEL_PATH}")
else:
    gdf = fetch_parcels(BASE_URL, LAYER_ID, FIELDS)
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels -> {PARCEL_PATH}")

for c in ["CurrentLand", "CurrentImpr", "CurrentTotal", "NetTaxes", "TIFDistrict"]:
    gdf[c] = pd.to_numeric(gdf[c], errors="coerce").fillna(0.0)

with_geom = gdf.geometry.notna().sum()
print(f"Parcels with geometry: {with_geom:,} ({with_geom/len(gdf)*100:.1f}%)")
print(f"Sum CurrentTotal: ${gdf['CurrentTotal'].sum():,.0f}")
print(gdf["PropertyClass"].value_counts(dropna=False))


In [ ]:
# Section 3 — Classify property types and flag exemptions

def classify(row):
    cls = str(row.get("PropertyClass", "") or "").strip()
    use = str(row.get("PropertyUse", "") or "").strip().lower()
    land = row.get("CurrentLand", 0) or 0
    impr = row.get("CurrentImpr", 0) or 0

    if cls == "Exempt":
        return "Exempt"
    # Vacant override: no building value (any use)
    if impr <= 0 and land > 0:
        return "Vacant Land"
    if "vacant" in use:
        return "Vacant Land"

    if cls.startswith("AG") or "agricultur" in use:
        return "Agricultural"
    if cls == "Manufacturing":
        return "Industrial"

    # Parking
    if "parking" in use:
        return "Transportation - Parking"

    # Residential uses
    if use == "single family":
        return "Single Family Residential"
    if use.startswith("condominium") or use.startswith("condo"):
        # split commercial-flavored condos out
        if any(k in use for k in ["office", "store", "retail", "warehouse", "apt"]):
            return "Office / Commercial Condo"
        return "Condominium"
    if use in ("2 unit", "3 unit") or use.startswith("2 unit") or use.startswith("3 unit"):
        return "Small Multi-Family (2-4 units)"
    if "apartment" in use or use.startswith("apts") or "rooming" in use:
        return "Large Multi-Family (5+ units)"

    # Commercial / other
    if "hotel" in use or "motel" in use:
        return "Hotel"
    if "office" in use or "bank" in use:
        return "Office / Commercial Condo"
    if any(k in use for k in ["store", "retail", "shop center", "shop,", "shop ", "mall",
                               "restaurant", "tavern", "gas", "supermarket", "grocery"]):
        return "Retail / General Commercial"
    if any(k in use for k in ["warehouse", "industrial", "manufactur"]):
        return "Industrial"
    if "mixed" in use or ("&" in use and ("apt" in use or "resid" in use)):
        return "Mixed Use"

    if cls == "Commercial":
        return "Commercial"
    if cls == "Residential":
        return "Other Residential"
    return "Other"

gdf["PROPERTY_CATEGORY"] = gdf.apply(classify, axis=1)

# Fully-exempt flag: exempt class or non-positive value
gdf["full_exmp"] = ((gdf["PropertyClass"].astype(str).str.strip() == "Exempt") |
                    (gdf["CurrentTotal"] <= 0)).astype(int)

# Taxable value columns (WI = 100% of market value, ratio 1.0)
gdf["taxable_land_value"] = gdf["CurrentLand"].clip(lower=0)
gdf["taxable_improvement_value"] = gdf["CurrentImpr"].clip(lower=0)
gdf["taxable_total"] = gdf["taxable_land_value"] + gdf["taxable_improvement_value"]

n_exempt = int(gdf["full_exmp"].sum())
print(f"Fully-exempt (held out): {n_exempt:,}")
print(f"Taxable parcels:         {(gdf['full_exmp']==0).sum():,}")
print("\nProperty category counts (all parcels):")
print(gdf["PROPERTY_CATEGORY"].value_counts())
print("\nResidual buckets check (should be small):")
for b in ["Other", "Other Residential", "Other Commercial"]:
    print(f"  {b}: {(gdf['PROPERTY_CATEGORY']==b).sum():,}")


In [ ]:
# Section 4 — Current tax (City of Madison MUNICIPAL portion), revenue-neutral to the certified levy
#
# The certified 2026 municipal levy ($326,666,400) was struck on the Jan 1 2025 roll; the
# open-data parcels are the newer Jan 1 2026 roll. Following the repo's Baltimore pattern we
# DERIVE the effective municipal millage on the current roll so total modeled municipal revenue
# equals the certified levy exactly (revenue-neutral to the City's real municipal levy).

taxable_mask = gdf["full_exmp"] == 0
taxable_base = gdf.loc[taxable_mask, "taxable_total"].sum()

DERIVED_CITY_MILLAGE = CERTIFIED_CITY_LEVY / taxable_base * 1000.0
print(f"Taxable base (2026 roll, non-exempt): ${taxable_base:,.0f}")
print(f"Derived effective municipal millage on current roll: {DERIVED_CITY_MILLAGE:.4f} mills")
print(f"(Published statutory rate on 2025 roll: {PUBLISHED_CITY_MILLAGE} mills — difference is the")
print(f" one-year assessment growth + TIF-increment inclusion, documented in the header.)")

gdf["millage_rate"] = DERIVED_CITY_MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col="taxable_total",
    millage_rate_col="millage_rate",
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    exemption_flag_col="full_exmp",
)

gap = (current_revenue / CERTIFIED_CITY_LEVY - 1) * 100
print(f"\nModeled municipal revenue: ${current_revenue:,.0f}")
print(f"Certified municipal levy:  ${CERTIFIED_CITY_LEVY:,.0f}")
print(f"Revenue gate: {gap:+.3f}%  (anchored to certified levy by construction)")
assert abs(gap) < 5.0, f"Revenue gap {gap:.2f}% exceeds 5%"

avg_home = 510_521  # avg single-family assessed value, 2026 roll (from source data)
print(f"\nAvg single-family city-portion tax on current roll @ derived rate: "
      f"${avg_home*DERIVED_CITY_MILLAGE/1000:,.0f}")


In [ ]:
# Section 5 — 4:1 split-rate model (revenue-neutral to the certified municipal levy)

# Exclude fully-exempt parcels from the solver AND the modeled output (they carry no signal).
df = gdf[gdf["full_exmp"] == 0].copy().reset_index(drop=True)
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts).")
print(f"Modeled taxable parcels: {len(df):,}")

land_millage, improvement_millage, new_revenue, df = model_split_rate_tax(
    df=df,
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    current_revenue=df["current_tax"].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"\nLand millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills  (ratio {land_millage/improvement_millage:.2f}:1)")
print(f"Revenue check:       ${new_revenue:,.0f}  (target ${df['current_tax'].sum():,.0f})")

category_summary = calculate_category_tax_summary(
    df=df, category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax", new_tax_col="new_tax",
)
print_category_tax_summary(category_summary,
                           title=f"Madison, WI — 4:1 Split-Rate (City municipal portion)")


In [ ]:
# Section 5b — Gate 5 artifact scan: read the results against economic priors

d = df.copy()
d["bldg_share"] = d["taxable_improvement_value"] / (
    d["taxable_land_value"] + d["taxable_improvement_value"]).replace(0, np.nan)
scan = d.groupby("PROPERTY_CATEGORY").agg(
    n=("tax_change_pct", "size"),
    median_pct=("tax_change_pct", "median"),
    median_bldg_share=("bldg_share", "median"),
    pct_zero_bldg=("taxable_improvement_value", lambda s: (s <= 1000).mean()),
).sort_values("median_pct", ascending=False)
print(scan.round(3))

# Headline shift stats
share_cut = (df["tax_change"] < 0).mean() * 100
share_inc = (df["tax_change"] > 0).mean() * 100
sf = df[df["PROPERTY_CATEGORY"] == "Single Family Residential"]
print(f"\nParcels with a CUT:      {(df['tax_change']<0).sum():,} ({share_cut:.1f}%)")
print(f"Parcels with an INCREASE: {(df['tax_change']>0).sum():,} ({share_inc:.1f}%)")
print(f"Single-family (owner-occupied proxy): n={len(sf):,}, "
      f"median change {sf['tax_change_pct'].median():+.1f}% / ${sf['tax_change'].median():+,.0f}")
print(f"Median SF current city tax ${sf['current_tax'].median():,.0f} -> new ${sf['new_tax'].median():,.0f}")


In [ ]:
# Section 6 — Exploration chart: median % change by category (quick visual)
fig, ax = plt.subplots(figsize=(10, 7))
med = df.groupby("PROPERTY_CATEGORY")["tax_change_pct"].median().sort_values()
colors = ["#228B22" if v < 0 else "#8B0000" for v in med.values]
med.plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Madison — Median City-Portion Tax Change by Category (4:1 Split-Rate)")
ax.set_xlabel("Median % change")
plt.tight_layout()
plt.savefig(DATA_DIR / "category_preview.png", dpi=150)
plt.close()
print("saved category_preview.png")


In [ ]:
# Section 7 — Census join (must happen before export)
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
if not os.getenv("CENSUS_API_KEY"):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in ["std_geoid", "median_income", "minority_pct", "black_pct"]:
        df[_col] = float("nan")
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                df = match_to_census_blockgroups(df, census_gdf)
                if "minority_pct" not in df.columns and "total_pop" in df.columns and "white_pop" in df.columns:
                    df["minority_pct"] = ((df["total_pop"] - df["white_pop"]) / df["total_pop"] * 100).round(2)
                if "black_pct" not in df.columns and "total_pop" in df.columns and "black_pop" in df.columns:
                    df["black_pct"] = (df["black_pop"] / df["total_pop"] * 100).round(2)
                print(f"Census join: {df['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in ["std_geoid", "median_income", "minority_pct", "black_pct"]:
                    df[_col] = float("nan")
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in ["std_geoid", "median_income", "minority_pct", "black_pct"]:
            df[_col] = float("nan")


In [ ]:
# Section 7b — Standard export, city report, parcel-map export
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f"../../analysis/data/{CITY_NAME}.csv",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    tax_change_col="tax_change",
    tax_change_pct_col="tax_change_pct",
    taxable_land_col="taxable_land_value",
    taxable_improvement_col="taxable_improvement_value",
    exempt_flag_col="full_exmp",
)

from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=df,
    city=CITY_NAME,
    output_path=f"../../analysis/maps/{CITY_NAME}.parquet",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
    property_category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    tax_change_col="tax_change",
    tax_change_pct_col="tax_change_pct",
    taxable_land_col="taxable_land_value",
    taxable_improvement_col="taxable_improvement_value",
    exempt_flag_col="full_exmp",
)
create_parcel_map(map_gdf, CITY_NAME, simplify_tolerance_m=2)
print("Done.")


## Validation Summary

- **Revenue gate (Gate 1):** modeled municipal revenue anchored to the certified 2026 City of Madison municipal levy **$326,666,400** (2026 Executive Operating Budget). Independent triangle check: published 7.0032 mills × official 2025-roll net-taxable base $46,627,007,300 = $326.5M (−0.05%).
- **Roll-year handling:** open-data parcels are the Jan 1 2026 roll; the certified levy was struck on the Jan 1 2025 roll, so the model uses the **derived effective municipal millage** on the current roll to hold the certified levy constant (Baltimore pattern). The split-rate distribution is roll-year invariant.
- **Scope:** City of Madison municipal portion only; all other taxing jurisdictions (Dane County, MMSD, MATC, state) held unchanged.
- **Exemptions:** preserved — fully-exempt parcels held out of the solver and excluded from the export/charts.
- **Gate 5 (artifact scan):** see Section 5b — building shares by category are read against priors (WI reports true land/improvement splits, so no $1,000-placeholder pathology expected).

### Wisconsin split-rate legal note
Wisconsin's **Uniformity Clause** (Wis. Const. art. VIII, § 1) requires that "the rule of taxation shall be uniform" for property, and Wisconsin courts read it strictly — real property generally cannot be split into classes taxed at different rates. A municipal land-value / split-rate tax would therefore **require enabling authority (a constitutional amendment or state legislation)** and is not available under current law. This is an **advocacy ask**, not an administrative option. (Flagged for the deck's legality section / `legality-analyzer`.)
